In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv('train.csv')

In [ ]:
df.head()

,id,date,country,store,product,num_sold
0,0,2010-01-01,Canada,Discount Stickers,Holographic Goose,NaN
1,1,2010-01-01,Canada,Discount Stickers,Kaggle,973.0
2,2,2010-01-01,Canada,Discount Stickers,Kaggle Tiers,906.0
3,3,2010-01-01,Canada,Discount Stickers,Kerneler,423.0
4,4,2010-01-01,Canada,Discount Stickers,Kerneler Dark Mode,491.0


In [ ]:
df.num_sold=df.num_sold.fillna(df.num_sold.median())

In [ ]:
df_=pd.read_csv('test.csv')

In [ ]:
from sklearn.preprocessing import LabelEncoder
for i in ['country','store','product']:
  le=LabelEncoder()
  df[i]=le.fit_transform(df[i])
  df_[i]=le.transform(df_[i])

In [ ]:
def createtimefeatures(df):
  df.date=pd.to_datetime(df.date)
  df['month'] = df['date'].dt.month
  df['day'] = df['date'].dt.day
  df['year'] = df['date'].dt.year
  df['quarter'] = df['date'].dt.quarter
  df['semester'] = np.where(df.quarter.isin([1,2]),1,2)
  df['dayofweek'] = df['date'].dt.dayofweek
  df['is_weekend'] = np.where(df['dayofweek'].isin([5,6]),1,0)
  return df

df=createtimefeatures(df)
df_=createtimefeatures(df_)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X,y=df.drop(columns=['date','num_sold']),df.num_sold

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=95)

In [ ]:
df.head()

,id,date,country,store,product,num_sold,month,day,year,quarter,semester,dayofweek,is_weekend
0,0,2010-01-01,0,0,0,605.0,1,1,2010,1,1,4,0
1,1,2010-01-01,0,0,1,973.0,1,1,2010,1,1,4,0
2,2,2010-01-01,0,0,2,906.0,1,1,2010,1,1,4,0
3,3,2010-01-01,0,0,3,423.0,1,1,2010,1,1,4,0
4,4,2010-01-01,0,0,4,491.0,1,1,2010,1,1,4,0


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

random_grid={'bootstrap': [True, False],
 'max_depth': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, None],
 'max_features': ['auto', 'sqrt'],
 'min_samples_leaf': [1, 2, 4],
 'min_samples_split': [2, 5, 10],
 'n_estimators': [200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000]}

rf=RandomForestRegressor(random_state=95)
rf_random = RandomizedSearchCV(estimator = rf, param_distributions = random_grid, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = -1)
rf_random.fit(X_train, y_train)


Fitting 3 folds for each of 100 candidates, totalling 300 fits


/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

The exit codes of the workers are {SIGKILL(-9)}

In [ ]:
model=rf_random.best_estimator_

In [ ]:
print(rf_random.best_params_)

In [ ]:
model.score(X_test,y_test)